# Forecast Q2 transactions

*Data Analysis in Python · Topic 10.2*

Your finance team plans stock from a transactions forecast, and they want to know how far off one usually is. Train on January 2024 to March 2025, forecast Q2 2025's weekly transactions, and score three forecasts on the weeks you held out.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell, the cell that builds the clean `sessions` table (lesson 3.'s steps) and the cell that builds the `weekly` table (lesson 6.'s steps): sessions, transactions and revenue per full Monday-to-Sunday week.
2. In the first cell marked ✏️, add two columns to the `weekly` table:
   - `t`: the week's number, from 0 for the first week
   - `week_of_year`: the week's ISO week number, 1 to 52, as a whole number

   Then create the `train` table, the weeks of the `weekly` table up to `TRAIN_END`, and the `test` table, the weeks after `TRAIN_END` up to `TEST_END`.
3. In the second cell marked ✏️, create `baseline_mape`: the MAPE of the same week last year as a forecast of the `test` table's transactions.
4. In the third cell marked ✏️, create `trend_mape`: the MAPE on the `test` table of a model of the log of transactions on `t` alone, trained on the `train` table.
5. In the last cell marked ✏️, create `seasonal_mape`: the same, for a model of the log of transactions on `t` and the week of the year as a category.
6. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

The MAPE is the mean of each week's absolute error, divided by that week's actual transactions, as a fraction.

**Expected output**: The `train` table has 65 weeks and the `test` table 13. `baseline_mape` is about `0.3239`, `trend_mape` about `0.5078` and `seasonal_mape` about `0.0806`.

## Parameters

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
MONTHS = [f"{year}-{month:02d}" for year in (2024, 2025) for month in range(1, 13)]
START, END = "2024-01-01", "2025-12-31"
TRAIN_END = "2025-03-30"
TEST_END = "2025-06-29"

In [ ]:
# The clean sessions table, with lesson 3's steps. Run it, and leave it as it is.
DAY_FIRST_MONTHS = ["2025-03"]
SOURCE_ALIASES = {
    "facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook",
    "lm.facebook.com": "facebook", "fb": "facebook",
    "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram",
}


def read_export(month):
    """Read one month's export with dates as dates, revenue as numbers and lost campaigns as missing."""
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    return export


raw = pd.concat([read_export(month) for month in MONTHS], ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["source"] = sessions["session_source"].str.lower().str.strip().replace(SOURCE_ALIASES)
sessions.shape

In [ ]:
# Sessions, transactions and revenue per full Monday-to-Sunday week, with lesson 6's steps. Run it, and leave it as it is.
daily = sessions.groupby("date")[["sessions", "transactions", "revenue"]].sum()
days_per_week = daily["sessions"].resample("W").count()
weekly = daily.resample("W").sum()
weekly = weekly[days_per_week == 7].copy()
weekly.shape

## The training and holdout weeks

In [ ]:
weekly["t"] = range(len(weekly))
weekly["week_of_year"] = weekly.index.isocalendar().week.astype(int)
train = weekly[weekly.index <= TRAIN_END]
test = weekly[(weekly.index > TRAIN_END) & (weekly.index <= TEST_END)]
len(train), len(test)

## The same week last year

In [ ]:
baseline = weekly["transactions"].shift(52).loc[test.index]
baseline_mape = ((baseline - test["transactions"]).abs() / test["transactions"]).mean()
baseline_mape

## A trend alone

In [ ]:
trend_model = smf.ols("np.log(transactions) ~ t", data=train).fit()
trend_forecast = np.exp(trend_model.predict(test))
trend_mape = ((trend_forecast - test["transactions"]).abs() / test["transactions"]).mean()
trend_mape

## A trend and the season

In [ ]:
model = smf.ols("np.log(transactions) ~ t + C(week_of_year)", data=train).fit()
forecast = np.exp(model.predict(test))
seasonal_mape = ((forecast - test["transactions"]).abs() / test["transactions"]).mean()
seasonal_mape

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    def number(value):
        return pd.api.types.is_number(value) and not isinstance(value, (pd.DataFrame, pd.Series))

    def near(value, *targets, tolerance=0.00001):
        return number(value) and any(abs(value - target) < tolerance for target in targets)

    weekly, train, test = found.get("weekly"), found.get("train"), found.get("test")
    if not isinstance(weekly, pd.DataFrame) or not {"t", "week_of_year"} <= set(weekly.columns):
        results.append((False, "The weekly table should have two more columns, t and week_of_year."))
    elif list(weekly["t"]) != list(range(len(weekly))):
        results.append((False, "t should number the weeks from 0: weekly[\"t\"] = range(len(weekly)), on the whole weekly table, before you split it."))
    elif not pd.api.types.is_integer_dtype(weekly["week_of_year"]) or str(weekly["week_of_year"].dtype) == "UInt32":
        results.append((False, f"week_of_year is {weekly['week_of_year'].dtype}. Make it a whole number with .astype(int): statsmodels can't read isocalendar()'s UInt32."))
    elif (weekly["week_of_year"].min(), weekly["week_of_year"].max()) != (1, 52) or weekly.loc["2025-11-30", "week_of_year"] != 48:
        results.append((False, "week_of_year should be the ISO week number of each week: weekly.index.isocalendar().week.astype(int). Black Friday week 2025, ending November 30, is week 48."))
    elif not isinstance(train, pd.DataFrame) or not isinstance(test, pd.DataFrame):
        results.append((False, "train and test should be DataFrames: the weekly table's rows up to TRAIN_END, and after it up to TEST_END."))
    elif len(train) == 104 or len(test) == 104:
        results.append((False, "train or test has all 104 weeks. Split the weekly table by date: weekly[weekly.index <= TRAIN_END] for train."))
    elif len(test) == 14:
        results.append((False, "The test table has 14 weeks: it starts with the week ending on TRAIN_END, which is also in train. Keep the weeks after TRAIN_END: weekly.index > TRAIN_END."))
    elif len(train) != 65 or len(test) != 13:
        results.append((False, f"train should have the 65 weeks up to March 30, 2025, and test the 13 weeks of Q2 2025. They have {len(train)} and {len(test)}."))
    elif "t" not in test.columns or test["t"].iloc[0] != 65:
        results.append((False, "The test table's t should continue from train's, 65 to 77. Add t to the weekly table before you split it."))
    else:
        results.append((True, f"Trained on {len(train)} weeks, to the week ending {train.index[-1]:%B} {train.index[-1].day}, {train.index[-1].year}, and held out the {len(test)} weeks of Q2 2025."))

    baseline = found.get("baseline_mape")
    if not number(baseline):
        results.append((False, f"baseline_mape should be a number: the mean of (baseline - actual).abs() / actual. It is {baseline!r}."))
    elif near(baseline, 32.385128, tolerance=0.001):
        results.append((False, "baseline_mape is 32.39: in percent. Keep it as a fraction, 0.3239."))
    elif not near(baseline, 0.323851):
        results.append((False, f"baseline_mape should be the MAPE of weekly[\"transactions\"].shift(52) on the test weeks: about 0.3239. It is {baseline:.4f}."))
    else:
        results.append((True, f"The same week last year is {baseline:.1%} off on average: the store grew a lot in a year."))

    trend = found.get("trend_mape")
    if not number(trend):
        results.append((False, f"trend_mape should be a number: the MAPE of np.exp(trend_model.predict(test)). It is {trend!r}."))
    elif near(trend, 0.98819, tolerance=0.001):
        results.append((False, "trend_mape is 0.988: the forecast is still in logs. Turn it back into transactions with np.exp(model.predict(test))."))
    elif not near(trend, 0.507760, 0.541082):
        results.append((False, f"trend_mape should be the MAPE of a model of np.log(transactions) on t, trained on train: about 0.5078. It is {trend:.4f}."))
    else:
        results.append((True, f"A trend alone is {trend:.1%} off: worse than last year's weeks, because it doesn't know that spring is quiet."))

    seasonal = found.get("seasonal_mape")
    if not number(seasonal):
        results.append((False, f"seasonal_mape should be a number: the MAPE of np.exp(model.predict(test)) for np.log(transactions) ~ t + C(week_of_year). It is {seasonal!r}."))
    elif near(seasonal, 0.039742, 0.041682):
        results.append((False, f"seasonal_mape is {seasonal:.4f}: too good, because the model trained on the test weeks. Fit it on the train table only."))
    elif near(seasonal, 0.089261):
        results.append((False, "seasonal_mape is 0.0893: the model isn't in logs, so the season is a fixed number of transactions that doesn't grow with the store. Use np.log(transactions) and np.exp() on the forecast."))
    elif near(seasonal, 0.507760, 0.541082):
        results.append((False, "seasonal_mape is the same as trend_mape. Add C(week_of_year) to the formula."))
    elif near(seasonal, 0.98819, 0.999, tolerance=0.02):
        results.append((False, "seasonal_mape is close to 1: the forecast is still in logs. Use np.exp(model.predict(test))."))
    elif not near(seasonal, 0.080645):
        results.append((False, f"seasonal_mape should be the MAPE of np.log(transactions) ~ t + C(week_of_year), trained on train: about 0.0806. It is {seasonal:.4f}."))
    else:
        results.append((True, f"A trend and the season are {seasonal:.1%} off in a typical week, against {found.get('baseline_mape', 0.3239):.1%} for the baseline."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
weekly["t"] = range(len(weekly))
weekly["week_of_year"] = weekly.index.isocalendar().week.astype(int)
train = weekly[weekly.index <= TRAIN_END]
test = weekly[(weekly.index > TRAIN_END) & (weekly.index <= TEST_END)]
len(train), len(test)
```

```python
baseline = weekly["transactions"].shift(52).loc[test.index]
baseline_mape = ((baseline - test["transactions"]).abs() / test["transactions"]).mean()
baseline_mape
```

```python
trend_model = smf.ols("np.log(transactions) ~ t", data=train).fit()
trend_forecast = np.exp(trend_model.predict(test))
trend_mape = ((trend_forecast - test["transactions"]).abs() / test["transactions"]).mean()
trend_mape
```

```python
model = smf.ols("np.log(transactions) ~ t + C(week_of_year)", data=train).fit()
forecast = np.exp(model.predict(test))
seasonal_mape = ((forecast - test["transactions"]).abs() / test["transactions"]).mean()
seasonal_mape
```

</details>